In [1]:
# 1. Import necessary libraries.
from flask import Flask, jsonify, make_response
from flask_cors import CORS
import requests
import time
import os

# 2. Create a Flask application instance.
app = Flask(__name__)

# 3. Apply CORS to the Flask app.
#    This is crucial for allowing the JavaScript Fetch API (from a file:// origin)
#    to make requests to our server (running on https://127.0.0.1:5000).
CORS(app)

# 4. Set up the OpenWeatherMap API Key.
API_KEY = "23e1c43e06ebbf18f7886c1f70760514" 

# 5. Initialize a dictionary for server-side caching.
#    This in-memory cache will store weather data to reduce redundant API calls.
server_cache = {}

# 6. Define a route to handle weather requests.
#    It accepts a dynamic 'city' parameter from the URL.
@app.route('/weather/<city>')
def get_weather(city):
    current_time = time.time()
    cache_duration = 600  # Cache validity period in seconds (10 minutes).

    # 6-1. [Server-Side Caching Logic] Check the cache first.
    #      If the city is in the cache and the data is not expired, return the cached data.
    if city in server_cache and current_time - server_cache[city]['timestamp'] < cache_duration:
        print(f" Returning cached data for '{city}'.")
        cached_data = server_cache[city]['data']
        
        # [Browser Caching Logic] Create a response and add the Cache-Control header.
        response = make_response(jsonify(cached_data))
        response.headers['Cache-Control'] = f'public, max-age={cache_duration}'
        return response

    # 6-2. [Proxy Logic] If not in cache, fetch new data from the OpenWeatherMap API.
    print(f"Fetching new data for '{city}' from the API.")
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    api_response = requests.get(url)

    # 6-3. Handle the API response.
    if api_response.status_code == 200:
        weather_data = api_response.json()
        
        # Store the new data in the server cache with a timestamp.
        server_cache[city] = {
            'data': weather_data,
            'timestamp': current_time
        }
        
        # [Browser Caching Logic] Add Cache-Control header to the new response.
        response = make_response(jsonify(weather_data))
        response.headers['Cache-Control'] = f'public, max-age={cache_duration}'
        return response
    else:
        # If the API call fails, return an error message.
        return jsonify({"error": "Failed to fetch weather from API"}), api_response.status_code

# 7. Run the Flask application.
#    This block executes only when the script is run directly.
if __name__ == '__main__':
    # Define paths for the SSL certificate and private key files.
    cert_file = 'cert.pem'
    key_file = 'key.pem'
    
    # Check if the required certificate files exist before starting.
    if not (os.path.exists(cert_file) and os.path.exists(key_file)):
        print("-" * 60)
        print(f"Error: Certificate files ('{cert_file}', '{key_file}') not found.")
        print("   Please generate them first using OpenSSL.")
        print("-" * 60)
    else:
        print("Starting the secure proxy server...")
        # Create an SSL context with the certificate and key.
        context = (cert_file, key_file)
        # Run the app on port 5000 with HTTPS enabled.
        # debug=False is recommended for stability in Jupyter Notebook.
        app.run(host='0.0.0.0', port=5000, ssl_context=context)

Starting the secure proxy server...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on https://127.0.0.1:5000
 * Running on https://10.30.0.200:5000
Press CTRL+C to quit


Fetching new data for 'Tokyo' from the API.


127.0.0.1 - - [14/Oct/2025 11:50:18] "GET /weather/Tokyo HTTP/1.1" 200 -
127.0.0.1 - - [14/Oct/2025 11:50:19] "GET /weather/Tokyo HTTP/1.1" 200 -


 Returning cached data for 'Tokyo'.


127.0.0.1 - - [14/Oct/2025 11:57:22] "GET /weather/Tokyo HTTP/1.1" 200 -


 Returning cached data for 'Tokyo'.
